In [8]:
import json
import openai
from urllib.request import Request, urlopen

client = openai.OpenAI()

URL = "https://nomad-movies.nomadcoders.workers.dev"

# 서버가 스크립트 요청을 차단할 수 있어 브라우저처럼 보이는 헤더 사용
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
}


def get_popular_movies():
    """URL의 경로 /movies 에서 인기 영화를 가져온다."""
    req = Request(f"{URL}/movies", headers=HEADERS)
    with urlopen(req) as resp:
        return json.loads(resp.read().decode())


def get_movie_details(movie_id):
    """URL의 경로 /movies/:id 에서 영화 정보를 가져온다."""
    req = Request(f"{URL}/movies/{movie_id}", headers=HEADERS)
    with urlopen(req) as resp:
        return json.loads(resp.read().decode())


def get_movie_credits(movie_id):
    """URL의 경로 /movies/:id/credits 에서 출연진 및 제작진을 가져온다."""
    req = Request(f"{URL}/movies/{movie_id}/credits", headers=HEADERS)
    with urlopen(req) as resp:
        return json.loads(resp.read().decode())


# 세 함수를 순서대로 실행 (id=550)
result_popular = get_popular_movies()
result_details = get_movie_details(550)
result_credits = get_movie_credits(550)

PROMPT = """
아래는 세 개의 함수를 순서대로 실행한 실제 결과야.

1. get_popular_movies() 결과:
{popular}

2. get_movie_details(550) 결과:
{details}

3. get_movie_credits(550) 결과:
{credits}

위 실행 결과를 순서대로 정리해서 요약해줘.
""".format(
    popular=json.dumps(result_popular, ensure_ascii=False, indent=2),
    details=json.dumps(result_details, ensure_ascii=False, indent=2),
    credits=json.dumps(result_credits, ensure_ascii=False, indent=2),
)

response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "user", "content": PROMPT},
    ]
)

message = response.choices[0].message.content

message

HTTPError: HTTP Error 403: Forbidden